In [1]:
from collections.abc import Iterable, Generator, Sequence
from dataclasses import dataclass
from math import tau, cos, sin, ceil, floor, log10, pow
from matplotlib import pyplot as plt
from matplotlib.ticker import MultipleLocator, FixedLocator
from itertools import batched
from functools import reduce, partial
from typing import TypeVar, Callable
import numpy as np

In [2]:
def samples_per_bit(sr: int, f: int, n: int) -> int:
    return (n * sr) // f


def to_nrz(bit: bool) -> int:
    return int(bit) * 2 - 1


def modulate_sample(a: float, cw: float, dw: float, t: float, dt: float) -> float:
    return a * cos(cw * t + dw * dt)


@dataclass
class Spec:
    sample_rate: int
    mark_frequency: int
    mark_num_periods: int
    mark_power_threshold_db: float
    space_frequency: int
    space_num_periods: int
    space_power_threshold_db: float
    low: int
    high: int

    @classmethod
    def with_kcs(cls) -> 'Spec':
        return Spec(9600, 2400, 8, 44, 1200, 4, -44, -127, 127)

    def bit_width(self) -> int:
        return samples_per_bit(self.sample_rate, self.mark_frequency, self.mark_num_periods)


def interpolate(steps, data: Sequence[int]) -> Generator[float, None, None]:
    n = steps * len(data)
    yield 0
    for i in range(1, n):
        index = int(ceil((i + 1) / steps)) - 1
        index_prev = int(ceil(i / steps)) - 1
        yield (data[index_prev] + data[index]) / 2


assert list(interpolate(2, list(range(0, 4)))) == [0, 0, 0.5, 1, 1.5, 2, 2.5, 3]


def integrate(data: Iterable[float]) -> Generator[float, None, None]:
    m = 0.0
    for d in data:
        m += d
        yield m


assert list(integrate([0, 3, 2])) == [0, 3, 5]


def modulate(spec: Spec, data: Iterable[bool]) -> Generator[int, None, None]:
    # Amplitude settings
    amplitude = (spec.high - spec.low) / 2

    # Frequency settings
    carrier_freq = (spec.mark_frequency + spec.space_frequency) // 2
    delta_freq = abs(spec.mark_frequency - spec.space_frequency) // 2
    sample_rate = spec.sample_rate
    carrier_omega = tau * (carrier_freq / sample_rate)
    delta_omega = tau * (delta_freq / sample_rate)

    window_size = spec.bit_width()

    nrz_data = (to_nrz(bit) for bit in data)
    interpolated_data = interpolate(window_size, list(nrz_data))
    integrated_data = integrate(interpolated_data)
    for i, m in enumerate(integrated_data):
        y = modulate_sample(amplitude, carrier_omega, delta_omega, float(i), m)
        yield int(y)


assert len(list(modulate(Spec.with_kcs(), [True, False]))) % Spec.with_kcs().bit_width() == 0


def center(signal: np.ndarray) -> np.ndarray:
    dc_offset = sum(signal) / len(signal)
    return signal - dc_offset


def normalize(signal: np.ndarray) -> np.ndarray:
    max_amplitude = max(abs(signal))
    return signal / max_amplitude


# Introduce zeros at the beginning and end, then add noise
def apply_noise(signal: np.ndarray, padding: int, noise_weight: float) -> np.ndarray:
    s = np.concat([np.zeros((padding,)), signal, np.zeros((padding,))])
    n = noise_weight * np.random.randn(s.shape[0])
    return s + n


def gen_bits(rng: np.random.Generator, n: int) -> np.ndarray:
    return rng.choice([np.bool(True), np.bool(False)], size=n)


def gen_normalized(signal: np.ndarray) -> np.ndarray:
    # Center and normalize the modulated data
    return normalize(center(signal))


def process(spec: Spec, rng: np.random.Generator, n: int, padding: int, noise_weight: float) -> tuple[np.ndarray, np.ndarray]:
    bits = gen_bits(rng, n)
    modulated = np.array(list(modulate(spec, bits)))
    noisy = apply_noise(modulated, padding, noise_weight)
    return (bits, gen_normalized(noisy))

In [5]:
spec = Spec.with_kcs()
rng = np.random.default_rng()

matrix = [
    (6, 0, 0.0),
    (6, 0, 0.1),
    (6, 20, 0.1),
    (128, 20, 0.1),
]

for n, p, w in matrix:
    bits, modulated = process(spec, rng, n, p, w)
    print(bits)

    with open(f'modulated-{n}bits-{p}padding-{w}noise.npz', mode='wb') as f:
        np.savez_compressed(f, source=bits, modulated=modulated, allow_pickle=False)

[ True  True False  True False  True]
[ True False  True False  True  True]
[ True  True False False False  True]
[ True False  True  True False False  True False  True  True False False
  True False False False False False False  True False False  True False
  True False  True False  True False False  True  True  True False False
 False  True False  True  True  True False  True False False False  True
 False  True False False  True False False False  True  True  True False
  True  True  True False False  True  True  True  True  True False False
  True  True  True False  True False  True False  True False  True False
 False False  True  True False  True False  True False  True  True  True
  True  True False  True False False False False False False False False
  True False  True False False False  True False False False False False
  True  True  True  True  True  True False False]
